# Lab 0: Document Preprocessor
**Goal:** turn a real regulatory PDF into clean, structured text with entities and keywords, ready to feed into Lab 3.

**Why:** every language-model system starts with text, and real documents are messy. If the text going in is broken, everything built on it (search, RAG, summaries) is worse. This lab uses the classic NLP toolkit from Module 0: cleaning, tokenisation, named-entity recognition and TF-IDF. **No API key and no paid service is needed.**

**Documents:** two real, public SEBI documents: a circular on retail algorithmic trading (February 2025) and a board memo on AI tools (December 2024). The notebook downloads them for you.

## Exercise 0: Setup
**Why we do this:** we need a PDF reader, a language-processing library (spaCy), a machine-learning library for TF-IDF (scikit-learn) and a plotting library.

**What the code does:** installs the packages, downloads spaCy's small English model (about 12 MB), and downloads the two PDFs into a `data/` folder.

**Key terms:** *spaCy* = an open-source library that splits text into tokens and tags words with grammar and entity information.

In [ ]:
%pip install -q pypdf spacy scikit-learn matplotlib nltk

In [ ]:
import os, re, json, urllib.request
import spacy.cli
try:
    import en_core_web_sm  # noqa: F401
except ImportError:
    spacy.cli.download("en_core_web_sm")
import spacy
nlp = spacy.load("en_core_web_sm")

SOURCES = {
    "sebi_retail_algo_circular_feb2025.pdf": "https://www.sebi.gov.in/sebi_data/attachdocs/feb-2025/1738665456458.pdf",
    "sebi_ai_responsibility_memo_dec2024.pdf": "https://www.sebi.gov.in/sebi_data/meetingfiles/dec-2024/1735042007618_1.pdf",
}
os.makedirs("data", exist_ok=True)
for name, url in SOURCES.items():
    if not os.path.exists(f"data/{name}"):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        open(f"data/{name}", "wb").write(urllib.request.urlopen(req).read())
print(os.listdir("data"))

**Expected output:** a list with the two PDF file names.

## Exercise 1: Load and clean the text
**Why we do this:** text pulled from a PDF is not the text you see on screen. It contains page headers, bullet symbols that come out as strange characters, curly quotes, and broken line breaks. Cleaning makes later steps reliable (Module 0, section 2).

**What the code does:** `load_pages` reads each page into a list of `(page_number, raw_text)`. First look at the raw text and count the odd characters. Then you write `clean_text`.

**Your task:** complete `clean_text(text)` so it:
1. removes page markers such as `Page 3 of 6`,
2. replaces the bullet character `\uf0b7` with `- `,
3. replaces curly quotes with straight quotes (`\u201c \u201d` become `"` and `\u2018 \u2019` become `'`),
4. collapses all runs of whitespace (including line breaks) into a single space.

**Important:** some words in these PDFs are split by the extraction itself (for example "mech anisms"). Simple cleaning cannot fix that reliably. Note it and move on; it is a known limit of PDF text extraction.

In [ ]:
from pypdf import PdfReader
from collections import Counter

def load_pages(path):
    return [(i + 1, p.extract_text() or "") for i, p in enumerate(PdfReader(path).pages)]

RAW = {name: load_pages(f"data/{name}") for name in SOURCES}
sample = RAW["sebi_retail_algo_circular_feb2025.pdf"][1][1]
print(repr(sample[:300]))

all_raw = " ".join(t for pages in RAW.values() for _, t in pages)
odd = Counter(ch for ch in all_raw if ord(ch) > 127)
print({hex(ord(ch)): n for ch, n in odd.most_common(6)})

def clean_text(text: str) -> str:
    # YOUR CODE HERE: apply the four cleaning steps above and return the cleaned string.
    raise NotImplementedError

<details><summary>Solution (try first)</summary>

```python
def clean_text(text):
    text = re.sub(r"Page \d+ of \d+", " ", text)
    text = text.replace("\uf0b7", "- ")
    text = text.replace("\u201c", '"').replace("\u201d", '"').replace("\u2018", "'").replace("\u2019", "'")
    return re.sub(r"\s+", " ", text).strip()
```
</details>

In [ ]:
CLEAN = {name: [(pg, clean_text(t)) for pg, t in pages] for name, pages in RAW.items()}
cleaned = " ".join(t for pages in CLEAN.values() for _, t in pages)
print(cleaned[:400])
print("\n'Page x of y' left:", len(re.findall(r"Page \d+ of \d+", cleaned)), "| bullet chars left:", cleaned.count("\uf0b7"))

**Expected output:** a readable opening paragraph with no line breaks, and `0` for both leftovers. The raw sample shows `\n` characters and page markers; the cleaned text should not.

**What to notice:** the odd-character counter shows exactly what needed fixing. Always inspect your data before writing cleaning rules, instead of guessing.

**Finance link:** a stray bullet or quote character can break a table lookup or a search for a clause. Clean input is the cheapest way to reduce downstream errors.

## Exercise 2: Tokenise and normalise
**Why we do this:** computers need text split into units and reduced to a standard form so that "regulation", "regulations" and "regulated" can be treated as related (Module 0, section 2).

**What the code does:** spaCy turns a sentence into *tokens*. Each token carries its *lemma* (dictionary form) and flags such as "is a stop word". We also compare it with *stemming*, a cruder method that chops word endings.

**Key terms:**
- *Token:* one unit of text, usually a word or punctuation mark.
- *Stop word:* a very common word ("the", "of") that carries little meaning.
- *Lemma:* the dictionary form of a word ("brokers" becomes "broker").
- *Stem:* a chopped-off root that may not be a real word ("regulation" becomes "regul").

**Your task:** complete `normalise(text)`. It should return a list of lowercase lemmas, keeping only alphabetic tokens that are not stop words. Hint: loop over `nlp(text)` and use `tok.lemma_`, `tok.is_alpha`, `tok.is_stop`.

In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

doc = nlp("Brokers shall tag all algo orders approved by the Exchanges.")
print([t.text for t in doc])
print(f"{'word':12}{'lemma':12}{'stem':10}")
for w in ["regulations", "brokers", "trading", "approved", "responsibility"]:
    t = nlp(w)[0]
    print(f"{w:12}{t.lemma_:12}{stemmer.stem(w):10}")

def normalise(text: str):
    # YOUR CODE HERE: return lowercase lemmas of alphabetic, non-stop-word tokens.
    raise NotImplementedError

<details><summary>Solution (try first)</summary>

```python
def normalise(text):
    return [t.lemma_.lower() for t in nlp(text) if t.is_alpha and not t.is_stop]
```
</details>

In [ ]:
page = CLEAN["sebi_retail_algo_circular_feb2025.pdf"][0][1]
terms = normalise(page)
print(len(page.split()), "words ->", len(terms), "normalised terms")
print(terms[:25])

**Expected output:** a word count dropping to roughly half as many terms, and a list of lowercase content words such as `circular`, `algorithmic`, `retail`, `investor`. The exact list depends on the spaCy model version.

**What to notice:** in the lemma and stem table, lemmas are always real words and stems are not. Stemming is faster but rougher; lemmatisation is usually better for readable results.

## Exercise 3: Find entities
**Why we do this:** extracting *who, when, how much and which rule* turns paragraphs into data you can search and check (Module 0, section 4).

**What the code does:** two approaches side by side.
- *Statistical NER (spaCy):* a trained model labels organisations, dates, money and so on. It handles varied wording but can miss domain-specific items.
- *Rules (regular expressions):* a pattern you write. It is exact for formats you know, such as circular numbers like `CIR/MRD/DP/09/2012`, but only finds what you describe.

**Key terms:** *NER* = named-entity recognition. *Regular expression* = a compact pattern for matching text shapes.

**Your task:** complete `regex_entities(text)` to return a dictionary with three lists: `circular_refs` (slash-separated reference numbers), `dates` (like `February 4, 2025`) and `amounts` (rupee amounts such as `Rs. 5 crore` or `\u20b91,250 crore`). Patterns are in the hints below.

In [ ]:
from collections import Counter

text = " ".join(t for _, t in CLEAN["sebi_retail_algo_circular_feb2025.pdf"])
ents = nlp(text[:6000]).ents
print(Counter(e.label_ for e in ents).most_common())
print([e.text for e in ents if e.label_ == "ORG"][:8])
print([e.text for e in ents if e.label_ == "DATE"][:8])

MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"

def regex_entities(text: str) -> dict:
    # YOUR CODE HERE: return {"circular_refs": [...], "dates": [...], "amounts": [...]}
    # circular_refs hint: r"\b[A-Z]{2,}(?:/[A-Za-z0-9\-]+){2,}\b"
    # dates hint:         rf"(?:{MONTHS})\s+\d{{1,2}},?\s+\d{{4}}"
    # amounts hint:       r"(?:Rs\.?|INR|\u20b9)\s?[\d,]+(?:\.\d+)?(?:\s?(?:crore|lakh|million|billion))?"
    raise NotImplementedError

<details><summary>Solution (try first)</summary>

```python
def regex_entities(text):
    return {
        "circular_refs": re.findall(r"\b[A-Z]{2,}(?:/[A-Za-z0-9\-]+){2,}\b", text),
        "dates": re.findall(rf"(?:{MONTHS})\s+\d{{1,2}},?\s+\d{{4}}", text),
        "amounts": re.findall(r"(?:Rs\.?|INR|\u20b9)\s?[\d,]+(?:\.\d+)?(?:\s?(?:crore|lakh|million|billion))?", text),
    }
```
</details>

In [ ]:
found = regex_entities(text)
print(found["circular_refs"])
print(found["dates"])
print(regex_entities("The fee is Rs. 5 crore and the limit is \u20b91,250 crore."))

**Expected output:** circular references including `SEBI/HO/MIRSD/MIRSD-PoD/P/CIR/2025/0000013` and `CIR/MRD/DP/09/2012`, a list of dates such as `February 4, 2025` and `March 30, 2012`, and the two rupee amounts from the test sentence. (The circulars themselves contain few rupee amounts, which is why we also test on a sentence.)

**What to notice:** compare the dates from spaCy with those from your pattern. Rules are precise for known formats; the model is more flexible. Real pipelines use both, and the circular reference numbers are something no general model will recognise.

**Finance link:** pulling out amounts, dates and reference numbers is the first step of KYC checks, loan-document review and regulatory tracking (Module 0, section 7).

## Exercise 4: TF-IDF, which terms are distinctive?
**Why we do this:** a word that appears everywhere ("SEBI", "shall") tells you little about one page. A word that appears often on one page and rarely elsewhere is *distinctive*. TF-IDF scores exactly that (Module 0, section 3).

**What the code does:** treats each page as a small document, then scores every term. *TF* (term frequency) rewards words repeated on a page. *IDF* (inverse document frequency) penalises words found on many pages. The product is the TF-IDF score.

**Key terms:** *Vectoriser* = a tool that turns text into numeric scores. *Bigram* = a two-word phrase such as "algo provider".

**Your task:** complete `top_terms(row, n)`. Hint: the vectoriser gives a matrix; take one row, sort its scores from high to low, and look up the term names with `vectorizer.get_feature_names_out()`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

PAGES = [(name, pg, t) for name, pages in CLEAN.items() for pg, t in pages if len(t) > 200]
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), lowercase=True)
matrix = vectorizer.fit_transform([t for _, _, t in PAGES])
terms_index = vectorizer.get_feature_names_out()
print(len(PAGES), "pages,", len(terms_index), "distinct terms")

def top_terms(row: int, n: int = 8):
    # YOUR CODE HERE: return a list of (term, score) for the n highest-scoring terms on page `row`.
    raise NotImplementedError

<details><summary>Solution (try first)</summary>

```python
def top_terms(row, n=8):
    scores = matrix[row].toarray().ravel()
    best = np.argsort(scores)[::-1][:n]
    return [(terms_index[i], round(float(scores[i]), 3)) for i in best]
```
</details>

In [ ]:
for row in (0, 3, len(PAGES) - 1):
    name, pg, _ = PAGES[row]
    print(f"{name} p.{pg}:", [t for t, _ in top_terms(row)])

terms, scores = zip(*top_terms(0, 8))
plt.barh(terms[::-1], scores[::-1])
plt.title("Most distinctive terms: first page"); plt.xlabel("TF-IDF score"); plt.tight_layout(); plt.show()

**Expected output:** a short list of distinctive terms per page and a bar chart. Terms specific to a page's topic should rank high, while terms used on every page should not.

**What to notice:** the same word can be common on one page and unremarkable overall. With only about a dozen small documents the scores are rough; TF-IDF gets more reliable with more text.

## Exercise 5: Produce structured output for Lab 3
**Why we do this:** the point of preprocessing is to hand later steps something clean and structured. Here you combine everything into one JSON file with a record per page.

**What the code does:** you write `process_page`, which returns a record with the document name, page number, cleaned text, entities (from spaCy and from your rules) and keywords (from TF-IDF). The cell then saves all records to `processed_documents.json`.

**Your task:** complete `process_page(row)`. Hint: take `name, pg, t = PAGES[row]`, run `nlp(t)` for entities, call `regex_entities(t)` and `top_terms(row)`.

**How Lab 3 uses this:** Lab 3 reads PDFs directly. As a stretch, change its loader to read the `text` field from this file and compare retrieval quality on cleaned versus raw text.

In [ ]:
def process_page(row: int) -> dict:
    # YOUR CODE HERE: return {"doc", "page", "text", "entities", "keywords"} for page `row`.
    # entities should be a dict: {"spacy": [{"text":..., "label":...}, ...], "rules": regex_entities(t)}
    raise NotImplementedError

<details><summary>Solution (try first)</summary>

```python
def process_page(row):
    name, pg, t = PAGES[row]
    return {
        "doc": name, "page": pg, "text": t,
        "entities": {
            "spacy": [{"text": e.text, "label": e.label_} for e in nlp(t).ents],
            "rules": regex_entities(t),
        },
        "keywords": [term for term, _ in top_terms(row)],
    }
```
</details>

In [ ]:
records = [process_page(i) for i in range(len(PAGES))]
with open("processed_documents.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(len(records), "records saved to processed_documents.json")
r = records[0]
print(r["doc"], "p.", r["page"], "| keywords:", r["keywords"][:5], "| circular refs:", r["entities"]["rules"]["circular_refs"])

**Expected output:** a record count equal to the number of pages kept, and a sample record with keywords and the circular reference found on page 1.

**What to notice:** open `processed_documents.json`. Every field came from a method you used in this lab. This is the shape of data that search, RAG and review workflows prefer to work with.

## Stretch goals
1. Fix the broken words ("mech anisms") by checking joined fragments against spaCy's vocabulary.
2. Add a rule that finds regulation names such as "Intermediaries Regulations, 2008".
3. Compare TF-IDF keywords with the keywords an LLM suggests for the same page. Which are more useful for search?

## Common errors
- **`NotImplementedError`:** you have not completed that exercise's function yet.
- **spaCy model not found:** re-run the setup cell, then restart the runtime.
- **A download fails (SEBI site):** retry, or download the PDFs in your browser and place them in `data/` under the same names.
- **Entity lists look different from the page:** spaCy models are statistical and change between versions. That is normal.